# Google Cloud Vision — OCR par API (référence commerciale)

Sert de référence pour comparer les modèles locaux. Test rapide sans code : déposez une image dans la démo
« Try the API » de https://cloud.google.com/vision#how-it-works.
**Chaque image est envoyée à Google** : n'utilisez pas de documents confidentiels sans accord.

| | |
|---|---|
| Modèle | [Cloud Vision API · DOCUMENT_TEXT_DETECTION](https://cloud.google.com/vision) |
| Licence | service payant (1 000 unités gratuites / mois, puis facturé à l'image) |
| Taille | modèle Google hébergé, rien à télécharger |
| Matériel | aucun : appel réseau |
| Sortie | texte complet + pages → blocs → paragraphes → mots avec boîtes |
| PDF | pages rastérisées avec `pypdfium2` (fonction commune `load_document`) |

Il faut un projet Google Cloud avec l'API Vision activée, puis **une** des deux authentifications :
`gcloud auth application-default login` (ou `GOOGLE_APPLICATION_CREDENTIALS=/chemin/cle.json`), ou une clé d'API
dans la variable `GOOGLE_API_KEY`. Le notebook se lance depuis le dossier `hf_ocr_notebooks/` (il importe `ocr_common.py`).

In [ ]:
%pip install -q google-cloud-vision pypdfium2 pillow pandas rapidfuzz datasets matplotlib

## 1. Jeux de données

Toutes les sources écrivent dans `data/<source>/` une image par échantillon et, à côté, sa vérité terrain `.txt` :

| Source | Contenu |
|---|---|
| `synthetic/` | 6 images + 1 PDF (français, anglais, tableau, petite police, rotation, bruit) |
| `malagasy/` | 5 images de phrases malgaches intégrées (sans réseau) |
| `hf_Lo-Renz-O_malagasy-sentence/` | phrases malgaches réelles du Hub, rendues en images (bruit / inclinaison) |
| `hf_Teklia_IAM-line/` | vraies images de lignes manuscrites anglaises (IAM) |
| `custom/` | vos documents (image ou PDF, + `nom.txt` optionnel pour mesurer le CER) |

Les datasets du Hub sont téléchargés une seule fois (paquet `datasets`) ; sans réseau, ils sont ignorés.

In [ ]:
import time
from pathlib import Path

from IPython.display import display

import ocr_common as oc

oc.make_test_set()
oc.make_malagasy_set()
for loader in (oc.load_hf_text_dataset, oc.load_hf_image_dataset):
    try:
        loader()
    except Exception as e:  # pas de réseau / pas de paquet datasets
        print(f"{loader.__name__} ignoré : {type(e).__name__}: {e}")

FILES = oc.list_inputs()
print(f"{len(FILES)} fichiers :", ", ".join(f"{f.parent.name}/{f.name}" for f in FILES))
oc.show_dataset([f for f in FILES if f.suffix != ".pdf"])

In [ ]:
import io
import os

DEVICE = "api"
# Indices de langue : [] = détection automatique. Essayez ["mg"] ou ["fr", "mg"] pour le malgache ;
# l'API renvoie une erreur si une langue n'est pas prise en charge.
LANGUAGE_HINTS = []
print("Matériel :", oc.hardware(), "| DEVICE =", DEVICE)

## 2. Chargement du modèle

In [ ]:
t0 = time.perf_counter()
from google.cloud import vision

api_key = os.environ.get("GOOGLE_API_KEY")
client = vision.ImageAnnotatorClient(client_options={"api_key": api_key} if api_key else None)
LOAD_S = time.perf_counter() - t0
print(f"Modèle chargé en {LOAD_S:.1f}s sur {DEVICE}")

## 3. Fonction d'extraction


`document_text_detection` est le mode adapté aux pages denses. Les boîtes affichées sont celles des paragraphes.

In [ ]:
def predict(image):
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    resp = client.document_text_detection(image=vision.Image(content=buf.getvalue()),
                                          image_context=vision.ImageContext(language_hints=LANGUAGE_HINTS))
    if resp.error.message:
        raise RuntimeError(resp.error.message)
    ann = resp.full_text_annotation
    boxes = [[(v.x, v.y) for v in para.bounding_box.vertices]
             for page in ann.pages for block in page.blocks for para in block.paragraphs]
    return ann.text, boxes


def extract(image):
    return predict(image)[0]

## 4. Validation visuelle

Pour chaque image : à gauche la page avec les zones détectées (en rouge), à droite la vérité terrain et le texte lu,
avec le CER dans le titre. Commencez par une image malgache, puis parcourez tout le jeu de test.

In [ ]:
img = oc.load_document(oc.MALAGASY_DIR / "mg_vaovao.png")[0]
t = time.perf_counter()
text, boxes = predict(img)
print(f"({time.perf_counter() - t:.1f}s)\n{text}")
oc.show_result(img, text, boxes, ref=oc.ground_truth(oc.MALAGASY_DIR / "mg_vaovao.png"), title="mg_vaovao.png")

In [ ]:
# toutes les sources sauf les PDF ; augmentez max_files pour tout voir
oc.visual_check(predict, [f for f in FILES if f.suffix != ".pdf"], max_files=12)

### PDF multi-pages

In [ ]:
pdf_path = oc.SYNTH_DIR / "rapport_2_pages.pdf"
for i, page in enumerate(oc.load_document(pdf_path), 1):
    t = time.perf_counter()
    text = extract(page)
    print(f"--- page {i} ({time.perf_counter() - t:.1f}s)\n{text}\n")

## 5. Benchmark sur tout le jeu de test

Temps par page et précision (CER / WER) sur tous les fichiers. Le premier appel inclut un éventuel temps de compilation/« chauffe ».
Les textes extraits sont enregistrés dans `results/<modèle>/` et les mesures dans `results/<modèle>.csv`
(utilisées par le notebook `00_comparaison.ipynb`).

In [ ]:
MODEL_NAME = "Google-Cloud-Vision"
df = oc.run_benchmark(extract, MODEL_NAME, FILES)
oc.save_meta(MODEL_NAME, repo="Cloud Vision API · DOCUMENT_TEXT_DETECTION", load_s=LOAD_S, device=str(DEVICE))
display(oc.summarize(df).to_frame(MODEL_NAME))

In [ ]:
display(df[[c for c in ("file", "type", "pages", "s_per_page", "cer", "wer", "error") if c in df]].round(3))
mg = df[df.file.str.startswith("mg_") | df.file.str.fullmatch(r"\d{3}\.png")]
if "cer" in mg and len(mg):
    print(f"CER moyen sur les échantillons malgaches et HF : {mg.cer.mean():.1%}")
oc.show_comparison(df, n=3)

## 6. Utiliser le modèle sur vos propres fichiers

Remplacez le chemin ci-dessous par une image ou un PDF ; le texte est enregistré à côté du fichier (`.ocr.txt`).

In [ ]:
def extract_file(path, dpi=200, max_pages=None):
    path = Path(path)
    pages = oc.load_document(path, dpi=dpi, max_pages=max_pages)
    text = "\n\n".join(extract(p) for p in pages)
    path.with_suffix(".ocr.txt").write_text(text, encoding="utf-8")
    return text


# print(extract_file("data/custom/mon_document.pdf"))